# 02 - Exploratory Data Analysis

This notebook answers the first business questions: how much was sold, when revenue was generated, which categories performed best, and how customers behaved.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_csv('../clean_csv/orders.csv', parse_dates=['order_date'])
customers = pd.read_csv('../clean_csv/customers.csv')
products = pd.read_csv('../clean_csv/products.csv')
returns = pd.read_csv('../clean_csv/returns.csv')
print({
    'orders': orders.shape,
    'customers': customers.shape,
    'products': products.shape,
    'returns': returns.shape
})

## Executive KPIs

In [ ]:
kpis = pd.Series({
    'Total orders': orders['order_id'].nunique(),
    'Total revenue (INR)': orders['order_total_inr'].sum(),
    'Average order value (INR)': orders['order_total_inr'].mean(),
    'Units sold': orders['quantity'].sum()
})
kpis

## Monthly revenue trend

In [ ]:
monthly_revenue = (orders.assign(month=orders['order_date'].dt.to_period('M').astype(str))
    .groupby('month', as_index=False)['order_total_inr'].sum())

monthly_revenue.plot(x='month', y='order_total_inr', figsize=(12, 4), marker='o', legend=False)
plt.title('Monthly Revenue')
plt.xlabel('Month')
plt.ylabel('Revenue (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
monthly_revenue.sort_values('order_total_inr', ascending=False).head()

## Revenue by category

In [ ]:
category_revenue = (orders.merge(products[['product_id', 'category']], on='product_id')
    .groupby('category', as_index=False)
    .agg(revenue_inr=('order_total_inr', 'sum'), units_sold=('quantity', 'sum'))
    .sort_values('revenue_inr', ascending=False))

category_revenue.plot.barh(x='category', y='revenue_inr', figsize=(9, 5), legend=False)
plt.title('Revenue by Product Category')
plt.xlabel('Revenue (INR)')
plt.ylabel('')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
category_revenue

## Customer behavior

In [ ]:
prime_summary = (orders.merge(customers[['customer_id', 'prime_member']], on='customer_id')
    .groupby('prime_member', as_index=False)
    .agg(customers=('customer_id', 'nunique'), orders=('order_id', 'nunique'), revenue_inr=('order_total_inr', 'sum')))

buyer_summary = (orders.groupby('customer_id')
    .agg(order_count=('order_id', 'nunique'), revenue_inr=('order_total_inr', 'sum'))
    .assign(buyer_segment=lambda x: x['order_count'].map(lambda n: 'One-time buyer' if n == 1 else 'Repeat buyer'))
    .groupby('buyer_segment', as_index=False)
    .agg(customers=('order_count', 'size'), orders=('order_count', 'sum'), revenue_inr=('revenue_inr', 'sum')))

display(prime_summary)
display(buyer_summary)

## Return reasons

This table helps identify the most common customer pain points before the diagnostic-analysis stage.

In [ ]:
return_reasons = (returns.groupby('return_reason', as_index=False)
    .agg(return_count=('return_id', 'nunique'), refund_inr=('refund_amount_inr', 'sum'))
    .sort_values('return_count', ascending=False))
return_reasons

## Key EDA takeaways

- Revenue is concentrated in a small number of product categories.
- Repeat buyers contribute most of the revenue, so retention is an important business lever.
- Prime and non-Prime customers can be compared by order count, revenue, and average order value.
- Return reasons identify the operational problems that should be investigated in the diagnostic analysis.
